<a href="https://colab.research.google.com/github/Lourdes-Churacutipa/Repositorio-de-lourdes/blob/main/Tarea5_Apellido_UHI_Arequipa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TAREA: Isla de Calor Urbana — Arequipa
## Maestría en Ciencia de Datos Ambientales | UTEC

**Basado en:** Lab6_Ses3_IslaCalor_GEE (Lima Metropolitana)
**Ciudad elegida:** Arequipa — ciudad andina árida, 2,335 m s.n.m.
**Comparación:** época seca (jun-ago) vs. verano austral (ene-mar), años 2025 y 2026 (FEN)

**Integrantes del grupo:** _completar_

---

### Nota metodológica sobre el AOI (urbano/periurbano)

A diferencia del ejemplo de Lima (que usaba una provincia entera como "zona urbana" y otra
provincia vecina como "zona periurbana"), aquí usamos un criterio más preciso:

1. **GAUL** nos da el límite administrativo de la **provincia de Arequipa** (nuestra AOI).
2. **MapBiomas Perú (Colección 3)** nos dice, píxel por píxel (30 m), qué hay realmente
   construido dentro de ese límite: usamos la clase **ID 24 = "Infraestructura urbana"**.
3. `zona_urbana` = píxeles clase 24 dentro de la provincia de Arequipa.
   `zona_periurbana` = píxeles NO urbanos (agrícola, matorral, afloramiento rocoso, etc.)
   dentro de la misma provincia — es decir, la periferia real, no una provincia distinta.

> **Supuesto:** MapBiomas Perú Colección 3 cubre hasta el año 2024 (banda `classification_2024`).
> Aún no existe banda 2025/2026, así que usamos 2024 como huella urbana de referencia — la
> forma construida de la ciudad no cambia de forma relevante en 1-2 años, así que sigue siendo
> válida para enmascarar las imágenes LST de 2025-2026.


---
## 1. Configuración de Google Earth Engine

In [ ]:
# Autenticación y conexión a GEE
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

try:
    ee.Initialize(project="TU-PROYECTO-GEE")  # <-- reemplaza con tu proyecto GEE
except Exception:
    ee.Authenticate()
    ee.Initialize(project="TU-PROYECTO-GEE")

print("GEE conectado correctamente")

---
## 2. Área de estudio — Provincia de Arequipa (Entregable 1)

Usamos **FAO GAUL** (igual que en el ejemplo de Lima) para delimitar la provincia,
y **MapBiomas Perú Col. 3** para distinguir urbano vs. periurbano dentro de ella.


In [ ]:
# Provincia de Arequipa — límites FAO GAUL Simplified 500m
gaul_l2 = (ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Peru")))

provincia_arequipa_fc = (gaul_l2
    .filter(ee.Filter.eq("ADM1_NAME", "Arequipa"))
    .filter(ee.Filter.eq("ADM2_NAME", "Arequipa")))

provincia_arequipa = provincia_arequipa_fc.geometry()

area_km2 = provincia_arequipa.area().divide(1e6).getInfo()
print(f"Provincia de Arequipa: {area_km2:.0f} km²")

In [ ]:
# MapBiomas Perú Colección 3 — huella urbana (clase 24 = Infraestructura urbana)
mapbiomas = ee.Image(
    "projects/mapbiomas-public/assets/peru/collection3/mapbiomas_peru_collection3_integration_v1"
)

BANDA_ANIO = "classification_2024"          # último año disponible en la Col. 3
CLASE_URBANA = 24                            # Infraestructura urbana (leyenda MapBiomas Perú Col. 3)

cobertura = mapbiomas.select(BANDA_ANIO).clip(provincia_arequipa)

urbano_mask = cobertura.eq(CLASE_URBANA)          # 1 = urbano, 0/masked = resto
periurbano_mask = cobertura.neq(CLASE_URBANA)     # todo lo NO urbano dentro de la provincia

# Área urbana detectada por MapBiomas (para verificar que el criterio tiene sentido)
area_urbana_km2 = (urbano_mask.selfMask()
    .multiply(ee.Image.pixelArea())
    .reduceRegion(ee.Reducer.sum(), provincia_arequipa, 30, maxPixels=1e10)
    .get(BANDA_ANIO).getInfo())
print(f"Área urbana (MapBiomas, clase 24) dentro de la provincia: {area_urbana_km2/1e6:.1f} km²")

In [ ]:
# Mapa de ubicación — AOI + huella urbana MapBiomas
Map0 = geemap.Map(center=[-16.4, -71.53], zoom=10, basemap="Esri.WorldTopoMap")
Map0.addLayer(provincia_arequipa, {"color": "gray"}, "Provincia de Arequipa (GAUL)")
Map0.addLayer(urbano_mask.selfMask(), {"palette": ["red"]}, "Zona urbana (MapBiomas clase 24)")
Map0.addLayer(periurbano_mask.selfMask(), {"palette": ["green"]}, "Zona periurbana (MapBiomas, resto)")
Map0

---
## 3. Fuente 1 — Landsat 8/9: Temperatura Superficial a 30 m

**Producto:** `LANDSAT/LC08/C02/T1_L2` y `LANDSAT/LC09/C02/T1_L2`

$$T_{\text{°C}} = (DN \times 0.00341802 + 149.0) - 273.15$$

Analizamos dos composiciones para **2025 y 2026**:
- **Verano austral** (ene-mar) → Entregable 2
- **Época seca** (jun-ago) → Entregable 3


In [ ]:
# Función para convertir ST_B10 a °C y enmascarar nubes
def procesar_landsat(img):
    qa = img.select("QA_PIXEL")
    mascara = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    lst = (img.select("ST_B10")
              .multiply(0.00341802)
              .add(149.0)
              .subtract(273.15)
              .updateMask(mascara)
              .rename("LST_C"))
    return lst.copyProperties(img, ["system:time_start"])

l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

def landsat_composite(mes_ini, mes_fin, anio_ini, anio_fin):
    col = (l8.merge(l9)
        .filterBounds(provincia_arequipa)
        .filter(ee.Filter.calendarRange(mes_ini, mes_fin, "month"))
        .filter(ee.Filter.calendarRange(anio_ini, anio_fin, "year"))
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .map(procesar_landsat))
    n = col.size().getInfo()
    return col.median(), n

# Verano (ene-mar) 2025-2026
landsat_verano, n_verano = landsat_composite(1, 3, 2025, 2026)
print(f"Imágenes Landsat verano 2025-2026: {n_verano}")

# Época seca (jun-ago) 2025-2026
landsat_seca, n_seca = landsat_composite(6, 8, 2025, 2026)
print(f"Imágenes Landsat época seca 2025-2026: {n_seca}")

### 3.1 Mapa LST — Verano (Entregable 2)

In [ ]:
vis_lst = {"min": 5, "max": 40, "palette": [
    "040274","040281","0502a3","0502b8","0602ff",
    "235cb1","307ef3","269db1","30c8e2","32d3ef",
    "3be285","3ff38f","86e26f","3ae237","b5e22e",
    "d6e21f","fff705","ffd611","ffb613","ff8b13",
    "ff6e08","ff500d","ff0000","de0101","c21301"
]}

Map1 = geemap.Map(center=[-16.4, -71.53], zoom=11, basemap="Esri.WorldTopoMap")
Map1.addLayer(landsat_verano.clip(provincia_arequipa), vis_lst, "LST Landsat 30m — Verano 2025-2026")
Map1.addLayer(urbano_mask.selfMask(), {"palette": ["red"]}, "Zona urbana (MapBiomas)")
Map1.add_colorbar(vis_lst, label="Temperatura superficial (°C)")
Map1

In [ ]:
# Tabla UHI — Verano
def uhi_tabla(imagen, escala, etiqueta):
    urb = (imagen.updateMask(urbano_mask)
           .reduceRegion(ee.Reducer.mean(), provincia_arequipa, escala, maxPixels=1e10)
           .get("LST_C").getInfo())
    peri = (imagen.updateMask(periurbano_mask)
           .reduceRegion(ee.Reducer.mean(), provincia_arequipa, escala, maxPixels=1e10)
           .get("LST_C").getInfo())
    uhi = urb - peri if urb is not None and peri is not None else None
    print(f"{etiqueta:25s} Urbana: {urb:6.2f} °C   Periurbana: {peri:6.2f} °C   UHI: {uhi:+.2f} °C")
    return {"periodo": etiqueta, "urbana_C": urb, "periurbana_C": peri, "UHI_C": uhi}

fila_verano = uhi_tabla(landsat_verano, 30, "Verano 2025-2026 (Landsat)")

### 3.2 Mapa LST — Época seca (Entregable 3)

In [ ]:
Map2 = geemap.Map(center=[-16.4, -71.53], zoom=11, basemap="Esri.WorldTopoMap")
Map2.addLayer(landsat_seca.clip(provincia_arequipa), vis_lst, "LST Landsat 30m — Época seca 2025-2026")
Map2.addLayer(urbano_mask.selfMask(), {"palette": ["red"]}, "Zona urbana (MapBiomas)")
Map2.add_colorbar(vis_lst, label="Temperatura superficial (°C)")
Map2

In [ ]:
fila_seca = uhi_tabla(landsat_seca, 30, "Época seca 2025-2026 (Landsat)")

tabla_uhi_landsat = pd.DataFrame([fila_verano, fila_seca])
tabla_uhi_landsat

**Opcional (recomendado para la sección 5 — efecto FEN):** repite `uhi_tabla` filtrando
solo 2025 y solo 2026 por separado (cambia `landsat_composite(..., 2025, 2025)` y
`(..., 2026, 2026)`) para ver si el año Niño (2026) muestra una UHI distinta.

In [ ]:
# Comparación año por año (2025 vs 2026) — para discutir el efecto del FEN
filas_anio = []
for anio in [2025, 2026]:
    img_v, _ = landsat_composite(1, 3, anio, anio)
    filas_anio.append({**uhi_tabla(img_v, 30, f"Verano {anio}"), "anio": anio, "estacion": "verano"})
    img_s, _ = landsat_composite(6, 8, anio, anio)
    filas_anio.append({**uhi_tabla(img_s, 30, f"Seca {anio}"), "anio": anio, "estacion": "seca"})

tabla_uhi_por_anio = pd.DataFrame(filas_anio)
tabla_uhi_por_anio

---
## 4. Fuente 2 — MODIS Terra: serie mensual de UHI (Entregable 4)

**Producto:** `MODIS/061/MOD11A2` — compuesto de 8 días, banda `LST_Day_1km`

$$T_{\text{°C}} = (DN \times 0.02) - 273.15$$

Calculamos la intensidad UHI **mes a mes** para 2025-2026 (en vez de la climatología
larga del ejemplo de Lima), para poder comparar directamente con el período de interés
de esta tarea (FEN incluido).


In [ ]:
def procesar_modis(img):
    lst = (img.select("LST_Day_1km")
              .multiply(0.02)
              .subtract(273.15)
              .rename("LST_C"))
    qc = img.select("QC_Day")
    mascara = qc.bitwiseAnd(3).eq(0)
    return lst.updateMask(mascara).copyProperties(img, ["system:time_start"])

def uhi_mensual_modis(mes, anio_ini=2025, anio_fin=2026):
    col = (ee.ImageCollection("MODIS/061/MOD11A2")
           .filterBounds(provincia_arequipa)
           .filter(ee.Filter.calendarRange(mes, mes, "month"))
           .filter(ee.Filter.calendarRange(anio_ini, anio_fin, "year"))
           .map(procesar_modis)
           .mean())
    urb = (col.updateMask(urbano_mask)
           .reduceRegion(ee.Reducer.mean(), provincia_arequipa, 1000, maxPixels=1e10)
           .get("LST_C").getInfo())
    peri = (col.updateMask(periurbano_mask)
           .reduceRegion(ee.Reducer.mean(), provincia_arequipa, 1000, maxPixels=1e10)
           .get("LST_C").getInfo())
    return {"mes": mes, "urb": urb, "peri": peri,
            "uhi": (urb - peri) if urb is not None and peri is not None else None}

print("Calculando serie MODIS 2025-2026 (puede tomar 1-2 min)...")
res_modis = [uhi_mensual_modis(m) for m in range(1, 13)]
df_uhi = pd.DataFrame(res_modis)

meses_es = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(range(1,13), df_uhi["urb"],  "ro-", label="Urbana")
axes[0].plot(range(1,13), df_uhi["peri"], "go-", label="Periurbana")
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(meses_es)
axes[0].set_ylabel("LST media (°C)"); axes[0].legend()
axes[0].set_title("LST mensual MODIS — Arequipa (2025-2026)")

axes[1].bar(range(1,13), df_uhi["uhi"],
            color=["salmon" if v and v > 0 else "steelblue" for v in df_uhi["uhi"]])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(meses_es)
axes[1].set_ylabel("Intensidad UHI (°C)")
axes[1].set_title("UHI mensual MODIS — Arequipa (2025-2026)")
plt.tight_layout(); plt.show()

print(f"Mes más caliente (ciudad) : {meses_es[df_uhi['urb'].idxmax()]}")
print(f"Mayor intensidad UHI      : {meses_es[df_uhi['uhi'].idxmax()]} ({df_uhi['uhi'].max():.2f} °C)")
df_uhi

---
## 5. Comparación: ¿la UHI es positiva o negativa según estación? ¿Por qué? (Entregable 5)

**Guía para completar (usa las tablas/gráfico de arriba):**

- Compara `fila_verano` vs. `fila_seca` (Landsat) y el patrón mensual de `df_uhi` (MODIS).
- ¿En qué meses la UHI es positiva (ciudad más caliente que el entorno) y en cuáles es
  negativa? En Lima, la UHI se vuelve **negativa en invierno** (la neblina costera enfría
  la ciudad mientras los cerros periurbanos reciben más sol) — ¿pasa algo parecido o
  distinto en Arequipa, que es una ciudad andina árida sin neblina costera?
- Considera factores propios de Arequipa: altitud (2,335 m), aridez, poca vegetación
  urbana, fuerte oscilación térmica día-noche típica de zonas andinas, cercanía a volcanes
  (El Misti, Chachani) que pueden influir en el entorno periurbano de referencia.
- Compara 2025 vs. 2026 (`tabla_uhi_por_anio`): ¿el año Niño (2026) altera la magnitud
  de la UHI respecto a 2025?

_(Completa aquí tu análisis en 1-2 párrafos, citando los números obtenidos.)_


---
## 6. Conclusión: ¿Cómo se compara con Lima? (Entregable 6)

**Guía para completar:**

- Lima: ciudad costera desértica, UHI positiva en verano y **negativa en invierno**
  (por la garúa/neblina que enfría más el centro urbano que la periferia agrícola).
- Arequipa: ciudad andina árida de altura, sin influencia marina directa.
- ¿La magnitud de la UHI en Arequipa es mayor, menor o similar a la de Lima?
- ¿El patrón estacional (signo de la UHI por estación) es el mismo o se invierte?
- ¿Qué rol juega la diferencia de resolución Landsat (30 m, detecta el patrón
  intraurbano) vs. MODIS (1 km, solo la tendencia mensual general) en tu interpretación?

_(Completa aquí tu conclusión final.)_


---
### Preguntas guía (para apoyar tu análisis en las secciones 5 y 6)

- ¿Tu ciudad también muestra UHI negativa en invierno, como Lima?
- ¿En qué mes es mayor la intensidad UHI? ¿Coincide con la estación seca/lluviosa local?
- ¿Qué diferencias hay entre Landsat (30 m) y MODIS (1 km)? ¿Cuál muestra mejor el
  patrón intraurbano?
